In [1]:
# 랭체인 도구로 에이전트 만들기
# 펑션 콜링은 함수를 호출하여 처리하는 기법이다
# 랭체인에서도 이와 유사한 방법을 도구(tools)라는 이름으로 제공한다.
# 랭체인의 도구 기능을 활용하면 자신이 만든 함수나 다른 사람이 만든 기능을 챗봇에 쉽게 추가할 수 있다.

# @tool 테코레이터로 랭체인에 함수 연결해보자.
# 펑션 콜링을 사용해서 만들었던 현재시간을 알려주는 챗봇을 랭체인으로 개발홰보자.

# @tool 테코레이터를 사용하면 함수를 도구로 변환할 수 있다.
# 이 데코레이터는 함수를 랭체인에서 외부 도구로 등록하여 언어 모델이 함수를 호출하고 사용할 수 있게 해준다.
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path(r"C:/LLM agent/Aiprojects/.env"))
api_key = os.getenv("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model="gpt-4o-mini", openai_api_key=api_key)

# llm.invoke([HumanMessage("잘 지냈어?")])
# invoke()는 체인을 실제로 실행해서 결과를 반환하는 함수

# invoke의 정확한 의미

# LangChain에서 모든 컴포넌트는 Runnable 인터페이스를 따름
# Runnable 인터페이스 "LangChain에서 모든 실행 가능한 객체를 통일된 방식으로 다루는 인터페이스"
# 그래서 공통 메서드가 있음:

# invoke() → 단일 입력 실행
# batch() → 여러 개 한 번에
# stream() → 스트리밍 출력

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

In [3]:
get_current_time.invoke({
    "timezone": "Asia/Seoul",
    "location": "서울"
})

Asia/Seoul (서울) 현재시각 2026-05-04 17:18:09 


'Asia/Seoul (서울) 현재시각 2026-05-04 17:18:09 '

In [ ]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_de7acce317', 'id': 'chatcmpl-DaElHZmYSn9GfYbm4hFx5p4HG6R16', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ddcfa-e462-78b1-9039-d038b331e80e-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_KD7ODfcWZ39Xp63W3y3cvv6k', 'type': 'tool_call'}],

In [5]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-04-30 15:02:09 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_de7acce317', 'id': 'chatcmpl-DaElHZmYSn9GfYbm4hFx5p4HG6R16', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ddcfa-e462-78b1-9039-d038b331e80e-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_KD7ODfcWZ39Xp63W3y3cvv6k', 'type': 'tool_call'}

In [6]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 2026년 4월 30일 오후 3시 2분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 192, 'total_tokens': 216, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_de7acce317', 'id': 'chatcmpl-DaElV5Cp4mRZObOxnmJZDDLvee0SR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ddcfb-1733-7e12-9d17-099446b223b9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 192, 'output_tokens': 24, 'total_tokens': 216, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})